# Recursive Confabulation — Reproduction Notebook

**Author:** Bentley DeVilling (Course Correct Labs)  
**Email:** Bentley@CourseCorrectLabs.com

This notebook reproduces the core findings from the Recursive Confabulation study. It:
- Installs pinned dependencies
- Loads CSV data from the repository
- Recomputes statistical tests
- Generates verification figures

Outputs are saved to `/content/figures/`


## 🚀 Quick Start

**To reproduce the full analysis:**
1. Click **Runtime → Run all** (or press Ctrl+F9 / Cmd+F9)
2. Wait for all cells to complete (~2-3 minutes)

**Manual execution:**
- Run cells sequentially from top to bottom

**Note:** If you encounter NumPy-related errors, go to **Runtime → Restart runtime**, then run all cells again.


In [ ]:
# Setup: Install dependencies in correct order
import sys, subprocess

# Install NumPy first (critical for binary compatibility)
print("📦 Installing NumPy...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4"])

# Then install packages that depend on NumPy
print("📦 Installing remaining dependencies...")
pkgs = ["pandas==2.2.2", "matplotlib==3.8.4", "scipy==1.12.0", "statsmodels==0.14.2"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Verify installations
print("\n✅ Verifying installations...")
import numpy, pandas, scipy, statsmodels, matplotlib
print(f"  numpy: {numpy.__version__}")
print(f"  pandas: {pandas.__version__}")
print(f"  scipy: {scipy.__version__}")
print(f"  statsmodels: {statsmodels.__version__}")
print(f"  matplotlib: {matplotlib.__version__}")
print("\n✅ All dependencies installed successfully!")


In [ ]:
# 2) Pull the repo so we have data/ and figures/ locally
import os, shutil, subprocess
REPO_URL = 'https://github.com/Course-Correct-Labs/recursive-confabulation.git'
REPO_DIR = '/content/recursive-confabulation'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
print('Repo cloned to', REPO_DIR)
os.makedirs('/content/figures', exist_ok=True)

In [ ]:
# 3) Load CSVs
import pandas as pd, os
DATA_DIR = os.path.join(REPO_DIR, 'data')
expected = ['harm_irr.csv', 'intervention_effects.csv', 'significance_matrix.csv']
available = [f for f in expected if os.path.exists(os.path.join(DATA_DIR, f))]
print('Found CSVs:', available)
dfs = {name: pd.read_csv(os.path.join(DATA_DIR, name)) for name in available}
for k,v in dfs.items():
    print(k, v.shape)
assert len(dfs) >= 1, 'No expected CSVs found under data/. Make sure data files are pushed to the repo.'

In [ ]:
# 4) Core statistical check example (Fisher exact), adjust column names to your schema
from scipy.stats import fisher_exact
import numpy as np

# Expect a significance matrix with columns like: condition, model, confab_n, correct_n
if 'significance_matrix.csv' in dfs:
    sm = dfs['significance_matrix.csv']
    # Example: pick one model-condition row group and compute Fisher on a 2x2 table
    # This is a template. Update the column names below to match your actual CSV headers.
    col_confab = 'confab_n'
    col_correct = 'correct_n'
    if col_confab in sm.columns and col_correct in sm.columns:
        # Make a quick 2-row comparison by splitting the first two rows
        if len(sm) >= 2:
            a = int(sm.iloc[0][col_confab]); b = int(sm.iloc[0][col_correct])
            c = int(sm.iloc[1][col_confab]); d = int(sm.iloc[1][col_correct])
            table = np.array([[a,b],[c,d]])
            _, p = fisher_exact(table)
            print('Fisher exact p-value (row0 vs row1):', p)
        else:
            print('significance_matrix.csv has < 2 rows. Skipping Fisher test example.')
    else:
        print('Expected columns confab_n and correct_n not found. Columns present:', list(sm.columns))
else:
    print('significance_matrix.csv not present. Skipping Fisher example.')

In [ ]:
# 5) Smoke-test figure: load any CSV and plot a simple bar chart
import matplotlib.pyplot as plt
import pandas as pd
import os

df_name = next(iter(dfs))
df = dfs[df_name]
plt.figure()
df.iloc[0:10,0].astype(str).value_counts().plot(kind='bar')
plt.title(f'Simple check on {df_name}')
out_path = '/content/figures/smoke_test_plot.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150)
print('Saved', out_path)
print('Done.')